# 00 — Data Extraction (Plan C)

Minari/D4RL Ant dataset에서 trajectory를 다운로드하고, Hilbert transform 기반 phase label과 quality filter를 적용해 `demos_ant_planC.npz`를 생성합니다.

이 노트북은 **실행 orchestration만 담당**합니다. 핵심 로직은 `src/data_extraction.py`, artifact 경로 관리는 `src/paths.py`에 있습니다.


## 1. Environment setup

Colab/로컬에서 필요한 system/package dependency를 설치합니다.

In [ ]:
!apt-get update -qq
!apt-get install -y libosmesa6-dev libgl1-mesa-glx libglfw3 patchelf --quiet
!pip install -q -r requirements.txt || pip install -q -r ../requirements.txt

## 2. Imports and project paths

In [ ]:
import os
import sys
from pathlib import Path

os.environ['MUJOCO_GL'] = 'osmesa'
os.environ['PYOPENGL_PLATFORM'] = 'osmesa'

REPO_ROOT_CANDIDATES = [Path.cwd(), Path.cwd().parent]
REPO_ROOT = next((p for p in REPO_ROOT_CANDIDATES if (p / 'src' / 'paths.py').exists()), None)
assert REPO_ROOT is not None, 'repo root with src/paths.py not found; run this notebook from the cloned repository'
SRC_DIR = REPO_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import minari
import numpy as np

from data_extraction import (
    DemoExtractionConfig,
    compare_phase_joint_candidates,
    extract_demos_from_episodes,
    list_ant_remote_datasets,
    load_first_available_minari_dataset,
    materialize_episodes,
    plot_demo_quality,
    print_demo_quality_report,
    save_demos,
)
from paths import ARTIFACT_ROOT, DATA_DIR, FIGURES_DIR, ensure_artifact_dirs

ensure_artifact_dirs()
print(f'Minari version: {minari.__version__}')
print(f'✓ src 경로 등록: {SRC_DIR}')
print(f'✓ artifact root: {ARTIFACT_ROOT}')

## 3. Dataset discovery and download

In [ ]:
ant_datasets = list_ant_remote_datasets(minari)
dataset, used_name = load_first_available_minari_dataset(minari)
assert dataset is not None, '사용 가능한 Ant dataset을 찾지 못했습니다. 후보 이름을 확인해 주세요.'

## 4. Episode diagnostics and phase-joint check

In [ ]:
config = DemoExtractionConfig()
episodes = materialize_episodes(dataset, expected_obs_dim=config.expected_obs_dim)
quality_summary = compare_phase_joint_candidates(
    episodes,
    hip_candidates=(13, 15, 17, 19),
    min_length=config.min_episode_length,
    smooth_sigma=config.smooth_sigma,
    dt=config.dt,
)
print(f'\nPhase 라벨링용 joint: obs[{config.phase_joint_idx}]')

## 5. Demo extraction and artifact save

In [ ]:
demos = extract_demos_from_episodes(episodes, config)
assert demos is not None, '선택된 demos가 없습니다. DemoExtractionConfig 임계값을 완화해 주세요.'
demos_path = save_demos(demos, DATA_DIR / 'demos_ant_planC.npz')

## 6. Quality plots and report

In [ ]:
demos_loaded = np.load(demos_path)
quality_path, visualization_path = plot_demo_quality(demos_loaded, FIGURES_DIR)
print_demo_quality_report(demos_loaded)
print('\n=== 00 Data Extraction 완료 ===')
print(f'Demos: {demos_path}')
print(f'Quality plot: {quality_path}')
print(f'Visualization: {visualization_path}')